In [1]:
import os,re,sys,shutil
import argparse
import numpy as np
from myfuncts import hasNum,find_nearest_ind,safe_log_array,replace_line
from myfuncts import load_stella_prof
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy.lib.recfunctions as rfn
M_sun = 1.98841e33 #grams

def load_blcode_prof(file_hyd, file_comp):
    dt_blcode_hyd=np.dtype([('ind',int),('m_center_g',float),
                      ('r_center_cm',float),('avg_rho',float), ('v_center_cmps',float),
                      ('Prad',float),('Q',float),('e_int',float),('Tavg',float),('n_e',float),
                      ('entropy',float)])
    dt_blcode_comp=np.dtype([('ind',int),('m_center_g',float),
                      ('r_center_cm',float),('h1',float),('he3',float),
                      ('he4',float),('c12',float),('n14',float),('o16',float),
                      ('ne20',float), ('mg24',float),('si28',float),
                      ('s32',float),('ar36',float),('ca40',float),('ti44',float),
                      ('cr48',float),('cr60',float),('fe52',float),('fe54',float),
                      ('fe56',float),('co56',float),('ni56',float)])  

    print("\nReading the input file %s\n" % (file_hyd))

    data_blcode_hyd = np.loadtxt(file_hyd,skiprows=2,dtype=dt_blcode_hyd)
    data_blcode_comp = np.loadtxt(file_comp,skiprows=3,dtype=dt_blcode_comp)
    abund_cols = list(dt_blcode_comp.names[3:]) 
    
    # 2. Merge the base hydro array with the sliced abundance array
    combined_data = rfn.merge_arrays(
        [data_blcode_hyd, data_blcode_comp[abund_cols]], 
        flatten=True, 
        usemask=False
    )

    return combined_data


In [2]:
blcode_profile_path = '/Users/beehuynh/Library/CloudStorage/Box-Box/workspace/BLcode/data/100M_blcode_rot_0pt5/Data_s100r0pt5_day175n17/'
lum_path = '/Users/beehuynh/Library/CloudStorage/Box-Box/workspace/BLcode/data/100M_blcode_rot_0pt5/lightcurve_blcode_100M_rot_0pt5_0pt1Zbase_combined_day175n17.dat'
blcode_hyd = 'hydrovars_nt_0000334332_time_316800.4759638'
blcode_comp = 'composition_nt_0000334332_time_316800.4759638'
outdir = '/Users/beehuynh/Library/CloudStorage/Box-Box/workspace/sim-superlite/data/blcode/100M_rot_0pt5_0pt1Zbase'
blcode_hyd_file = blcode_profile_path + blcode_hyd
blcode_abn_file = blcode_profile_path + blcode_comp
data_hyd = load_blcode_prof(blcode_hyd_file, blcode_abn_file)
data_print_hyd = load_blcode_prof(blcode_hyd_file, blcode_abn_file)


Reading the input file /Users/beehuynh/Library/CloudStorage/Box-Box/workspace/BLcode/data/100M_blcode_rot_0pt5/Data_s100r0pt5_day175n17/hydrovars_nt_0000334332_time_316800.4759638


Reading the input file /Users/beehuynh/Library/CloudStorage/Box-Box/workspace/BLcode/data/100M_blcode_rot_0pt5/Data_s100r0pt5_day175n17/hydrovars_nt_0000334332_time_316800.4759638



In [46]:
print(data_hyd)

[(  1, 1.066713e+35, 6.151401e+13, 1.226377e-09, 3.428333e+07, 4.800679e+03, 0.000000e+00, 9.434892e+12, 32783.74  , 0.5, 0., 5.391994e-71, 3.336681e-61, 0.00147671, 0.1464109, 9.094337e-20, 0.7646339, 0.07676165, 0.01070423, 1.263828e-05, 5.397111e-11, 1.529844e-18, 5.113020e-28, 2.498753e-39, 6.721868e-52, 1.036331e-99, 2.525712e-66, 1.036331e-99, 1.755436e-86, 1.547374e-84, 4.985505e-84)
 (  2, 1.066902e+35, 6.183670e+13, 1.034460e-09, 3.438127e+07, 3.931351e+03, 0.000000e+00, 9.198761e+12, 31274.05  , 0.5, 0., 7.620714e-71, 3.842834e-61, 0.00152725, 0.1464415, 1.067805e-19, 0.7646421, 0.0766806 , 0.01069602, 1.257685e-05, 5.343740e-11, 1.504408e-18, 4.987687e-28, 2.418974e-39, 6.453547e-52, 1.036331e-99, 2.402571e-66, 1.036331e-99, 1.654819e-86, 1.458683e-84, 4.699745e-84)
 (  3, 1.067092e+35, 6.221495e+13, 8.290873e-10, 3.451514e+07, 2.986673e+03, 0.000000e+00, 8.746673e+12, 29257.26  , 0.5, 0., 9.631693e-71, 4.311195e-61, 0.00157055, 0.1464709, 1.214550e-19, 0.7646495, 0.07660869

In [4]:
# -- Load luminosity data
dt_terminal =np.dtype([('time',float),('L_bol',float),
                      ('r_phot',float),('delay_day',float), ('time_s',float), ('tau',float)])
terminal_data = np.genfromtxt(lum_path, dtype=dt_terminal, skip_header=2)

blcode_hyd_file = blcode_profile_path + blcode_hyd
blcode_abn_file = blcode_profile_path + blcode_comp
# Obtain time of hydrovars profile
with open(blcode_hyd_file) as f:
    first_line = f.readline().strip()
time_str = first_line.split()[-1]
time_s = float(time_str)


In [3]:
L_bol = terminal_data['L_bol'][find_nearest_ind(terminal_data['time_s'], time_s)]
print(time_s, '\t', L_bol)

NameError: name 'terminal_data' is not defined

In [5]:
total_mass_ejecta = (data_hyd['m_center_g'][-1]-data_hyd['m_center_g'][0])/M_sun
print(total_mass_ejecta, '\t', data_hyd['m_center_g'][-1], data_hyd['m_center_g'][0])

7.078117692025293 	 1.207455e+35 1.066713e+35


In [35]:
(1.207455E+35-1.066713E+35)/ 1.98841e33 

7.078117692025293

In [36]:
print(data_hyd['ind'])

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 239 240 241 242 243 244 245 246 24

In [42]:
import glob
tau_path = '/Users/beehuynh/Library/CloudStorage/Box-Box/workspace/BLcode/data/100M_blcode_rot_0pt5/tau_combined_day175n17/*'
files = sorted(glob.glob(tau_path))
for file in files:
    dt_tau = np.dtype([('r_centre[cm]',float),('tau_centre',float)])
    with open(file, 'r') as f_tau:
        for line in f_tau:
            if line.startswith('# Filename:'):
                tau_filename = line.split(':')[1].strip()
            if not line.startswith('#'):
                break
        if tau_filename == blcode_hyd:
            print("Found the tau file for the hydrovars profile!", tau_filename)
            data_tau = np.genfromtxt(file, dtype=dt_tau, skip_header=1)
            break

Found the tau file for the hydrovars profile! hydrovars_nt_0000000000_time_0.0000000


In [3]:
time_s = 1.547280e+07/24/3600
day = round(time_s, 2)
day, str(day)

(179.08, '179.08')